# Module 12: Wednesday Session Demo
## CS 5045: Computation for the Data Sciences
**Text Analytics and NLP: Live Demo**

This notebook walks through the text analytics pipeline on a small inline product review corpus.
For each step, we name the computational operation *before* writing the code.

**Corpus:** 10 product reviews defined inline below. No external file needed.

| Step | Operation | What We Do |
|------|-----------|------------|
| 1 | Represent | Build the corpus as a DataFrame |
| 2 | Transform | Normalize: lowercase and remove punctuation |
| 3 | Transform | Tokenize: split each review into a word list |
| 4 | Filter | Remove stop words |
| 5 | Aggregate | Build the word frequency table |
| 6 | Transform | Score sentiment per document |


---
## Represent: Build the Corpus

We define the reviews inline as a list of dictionaries, then load them into a DataFrame.
The same code would work if we swapped the inline list for `pd.read_csv(...)` on a real file;
the rest of the pipeline does not change.


In [1]:
import pandas as pd
import re
from collections import Counter

# REPRESENT: corpus defined inline; no external file required
reviews = [
    {"product": "headphones",   "rating": 5, "review_text": "Amazing sound quality. Very comfortable, great bass and the battery lasts forever."},
    {"product": "headphones",   "rating": 2, "review_text": "Broke after two weeks. Terrible build quality and the left ear cracked."},
    {"product": "coffee_maker", "rating": 5, "review_text": "Best coffee maker I have ever owned. Brews quickly and the carafe stays hot."},
    {"product": "coffee_maker", "rating": 1, "review_text": "Leaks everywhere. Worst purchase of the year. Completely worthless."},
    {"product": "backpack",     "rating": 4, "review_text": "Sturdy and lightweight. Comfortable straps and lots of pockets. Love it."},
    {"product": "backpack",     "rating": 2, "review_text": "Flimsy zippers and the seams are already fraying. Disappointed after one month."},
    {"product": "scale",        "rating": 5, "review_text": "Accurate readings every time. Smooth surface and easy to clean. Excellent."},
    {"product": "scale",        "rating": 3, "review_text": "Readings are inconsistent. Not terrible but not great either. Average product."},
    {"product": "charger",      "rating": 4, "review_text": "Charges quickly and the grip is solid. Works perfectly with my phone."},
    {"product": "charger",      "rating": 1, "review_text": "Awful. Smells like burning plastic and drains the battery instead of charging it."},
]

df = pd.DataFrame(reviews)
print(f"Loaded {len(df)} reviews across {df['product'].nunique()} products")
print(f"Columns: {list(df.columns)}")
df

Loaded 10 reviews across 5 products
Columns: ['product', 'rating', 'review_text']


,product,rating,review_text
0,headphones,5,"Amazing sound quality. Very comfortable, great..."
1,headphones,2,Broke after two weeks. Terrible build quality ...
2,coffee_maker,5,Best coffee maker I have ever owned. Brews qui...
3,coffee_maker,1,Leaks everywhere. Worst purchase of the year. ...
4,backpack,4,Sturdy and lightweight. Comfortable straps and...
5,backpack,2,Flimsy zippers and the seams are already frayi...
6,scale,5,Accurate readings every time. Smooth surface a...
7,scale,3,Readings are inconsistent. Not terrible but no...
8,charger,4,Charges quickly and the grip is solid. Works p...
9,charger,1,Awful. Smells like burning plastic and drains ...


---
## Transform: Normalize the Text

Two steps before we can count anything:

1. **Lowercase:** `'Great'` and `'great'` must count as the same word.
2. **Remove punctuation:** `'quality.'` and `'quality'` must count as the same word.

> **Pause:** What would happen to the word counts if we skipped the lowercase step?
> Run the cell and look at the before/after side by side.


In [2]:
# TRANSFORM step 1: lowercase
df['review_clean'] = df['review_text'].str.lower()

# TRANSFORM step 2: remove punctuation
df['review_clean'] = df['review_clean'].apply(
    lambda text: re.sub(r'[^a-z\s]', '', text)
)

# Before and after
df[['review_text', 'review_clean']].head(4)

,review_text,review_clean
0,"Amazing sound quality. Very comfortable, great...",amazing sound quality very comfortable great b...
1,Broke after two weeks. Terrible build quality ...,broke after two weeks terrible build quality a...
2,Best coffee maker I have ever owned. Brews qui...,best coffee maker i have ever owned brews quic...
3,Leaks everywhere. Worst purchase of the year. ...,leaks everywhere worst purchase of the year co...


---
## Transform: Tokenize

**Tokenization** splits each cleaned string into a list of individual words.
In Python this is `.split()`: a method you already know.

Once each review is a list of words, Filter and Aggregate can work on it directly.

> **Pause:** Before tokenizing, what type is each cell in `review_clean`?  
> After tokenizing, what type is each cell in `tokens`?


In [3]:
# TRANSFORM: split each cleaned review into a list of words
df['tokens'] = df['review_clean'].str.split()

print('Before tokenizing:', df['review_clean'].iloc[0])
print()
print('After tokenizing: ', df['tokens'].iloc[0])

Before tokenizing: amazing sound quality very comfortable great bass and the battery lasts forever

After tokenizing:  ['amazing', 'sound', 'quality', 'very', 'comfortable', 'great', 'bass', 'and', 'the', 'battery', 'lasts', 'forever']


---
## Filter: Remove Stop Words

**Stop words** are common English words (`'the'`, `'a'`, `'and'`) that carry grammar
but not meaning. Removing them is a **Filter** applied within each token list.

> **Pause:** Run the first cell below to see the top 10 words *before* stop word removal.
> What do you expect to see? Then run the second cell to filter and compare.


In [4]:
# Word counts BEFORE stop word removal: see the problem
all_raw = []
for tlist in df['tokens']:
    all_raw.extend(tlist)

print('Top 10 words BEFORE stop word removal:')
for word, count in Counter(all_raw).most_common(10):
    print(f'  {word:<20} {count}')

Top 10 words BEFORE stop word removal:
  and                  9
  the                  7
  of                   3
  quality              2
  comfortable          2
  great                2
  battery              2
  after                2
  terrible             2
  quickly              2


In [5]:
# FILTER: define stop words and remove them from each review's token list
STOP_WORDS = {
    'the', 'a', 'an', 'and', 'is', 'in', 'it', 'to', 'for',
    'of', 'my', 'i', 'at', 'this', 'very', 'but', 'after',
    'even', 'are', 'not', 'do', 'from', 'be', 'all', 'have',
    'too', 'was', 'on', 'up', 'so', 'one', 'has', 'me',
    'they', 'get', 'its', 'that', 'with', 'what', 'no',
    'like', 'already', 'instead', 'either', 'lots'
}

df['tokens_filtered'] = df['tokens'].apply(
    lambda words: [w for w in words if w not in STOP_WORDS]
)

# Compare before and after for the first review
print('Before filter:', df['tokens'].iloc[0])
print()
print('After filter: ', df['tokens_filtered'].iloc[0])

Before filter: ['amazing', 'sound', 'quality', 'very', 'comfortable', 'great', 'bass', 'and', 'the', 'battery', 'lasts', 'forever']

After filter:  ['amazing', 'sound', 'quality', 'comfortable', 'great', 'bass', 'battery', 'lasts', 'forever']


---
## Aggregate: Word Frequency Table

**Flatten** all token lists into one big list, then **count** with Counter.

This is Aggregate: collapsing many tokens across many documents into one frequency summary.
The result is called a **bag of words**: no order, no grammar, just counts.

> **Pause:** Look at the top 10 words after stop word removal.
> What story do they tell about customer experience with these products?


In [6]:
# AGGREGATE: flatten all token lists and count
all_tokens = []
for tlist in df['tokens_filtered']:
    all_tokens.extend(tlist)

word_counts = Counter(all_tokens)

print('Top 10 words AFTER stop word removal:')
for word, count in word_counts.most_common(10):
    bar = '#' * count
    print(f'  {word:<20} {count:3d}  {bar}')

Top 10 words AFTER stop word removal:
  quality                2  ##
  comfortable            2  ##
  great                  2  ##
  battery                2  ##
  terrible               2  ##
  quickly                2  ##
  readings               2  ##
  amazing                1  #
  sound                  1  #
  bass                   1  #


---
## Quick Check: Counter in a Terminal (30 seconds)

Open a terminal (or the VS Code terminal panel) and paste this one command.
It shows that Counter works identically outside a notebook: no import changes, no special setup.

```
python -c "from collections import Counter; words = 'the quick brown fox jumped over the lazy dog'.split(); print(Counter(words).most_common(5))"
```

If your project code runs as a script, the same Counter pattern applies with no modifications.


---
## Transform: Sentiment Score per Document

Instead of using the star rating to define sentiment, we use the *words themselves*.

**Logic per document:**
- Count how many tokens appear in a set of positive words (`pos_score`)
- Count how many tokens appear in a set of negative words (`neg_score`)
- Compare the two scores; the higher one wins

This is a Transform: each document goes in, a label comes out.

> **Pause:** Before running, predict: which review do you think will score highest
> on positive words? Which will score highest on negative words?


In [7]:
# TRANSFORM: keyword-based sentiment scoring
POSITIVE_WORDS = {
    'great', 'excellent', 'amazing', 'comfortable', 'best', 'perfect',
    'love', 'awesome', 'smooth', 'accurate', 'lightweight', 'fantastic',
    'sturdy', 'clear', 'quickly', 'grip', 'solid'
}

NEGATIVE_WORDS = {
    'broke', 'terrible', 'worst', 'disappointed', 'cracked', 'flimsy',
    'weak', 'worthless', 'leaks', 'awful', 'drains', 'fraying', 'inconsistent'
}

def classify_review(tokens):
    """TRANSFORM: return a sentiment label for one review's token list."""
    pos_score = sum(1 for w in tokens if w in POSITIVE_WORDS)
    neg_score = sum(1 for w in tokens if w in NEGATIVE_WORDS)
    if pos_score > neg_score:
        return 'positive'
    elif neg_score > pos_score:
        return 'negative'
    else:
        return 'neutral'

# Apply to each row
df['pos_score'] = df['tokens_filtered'].apply(
    lambda tokens: sum(1 for w in tokens if w in POSITIVE_WORDS)
)
df['neg_score'] = df['tokens_filtered'].apply(
    lambda tokens: sum(1 for w in tokens if w in NEGATIVE_WORDS)
)
df['sentiment'] = df['tokens_filtered'].apply(classify_review)

# Show results
print('Sentiment distribution (keyword-based):')
print(df['sentiment'].value_counts())
print()

# Compare keyword sentiment to star rating
print('Document-level scores:')
cols = ['product', 'rating', 'pos_score', 'neg_score', 'sentiment']
print(df[cols].to_string(index=False))

Sentiment distribution (keyword-based):
sentiment
positive    5
negative    5
Name: count, dtype: int64

Document-level scores:
     product  rating  pos_score  neg_score sentiment
  headphones       5          3          0  positive
  headphones       2          0          3  negative
coffee_maker       5          2          0  positive
coffee_maker       1          0          3  negative
    backpack       4          4          0  positive
    backpack       2          0          3  negative
       scale       5          3          0  positive
       scale       3          1          2  negative
     charger       4          3          0  positive
     charger       1          0          2  negative


In [8]:
# FILTER by keyword: change 'comfortable' to any word in the corpus
keyword = 'comfortable'
matches = df[df['review_clean'].str.contains(keyword)]
print(f'Reviews mentioning "{keyword}":')
print(matches[['product', 'rating', 'sentiment', 'review_text']].to_string(index=False))

Reviews mentioning "comfortable":
   product  rating sentiment                                                                        review_text
headphones       5  positive Amazing sound quality. Very comfortable, great bass and the battery lasts forever.
  backpack       4  positive           Sturdy and lightweight. Comfortable straps and lots of pockets. Love it.


---
## Connecting to hw12.ipynb

The assignment applies this same pipeline to the full 260-review product dataset (`data.csv`).
Sections 1 to 5 ask you to fill in the key expressions; the operations are identical to what you just ran.
Section 6 has no blanks, and Section 7 is a short C exercise.

| hw12 Section | Operations | Key blank |
|--------------|------------|-----------|
| Section 1 | Represent | `pd.read_csv(...)` |
| Section 2 | Transform | `str.lower()`, `re.sub(...)`, `str.split()` |
| Section 3 | Filter | `[w for w in words if w not in STOP_WORDS]` |
| Section 4 | Aggregate | `Counter(all_tokens)`, `.most_common()` |
| Section 5 | Transform + Filter + Aggregate | `classify_review` using `sum(1 for w in tokens if ...)` |
| Section 6 | Filter + Aggregate (Code From Scratch) | no blanks: split by median `helpful_votes`, compare top-5 words |
| Section 7 | Back to C | fix the case comparison in `word_match.c` |

**The one pattern that may look new in Section 5:**

```python
pos_score = sum(1 for w in tokens if w in POSITIVE_WORDS)
```

This adds 1 for each token that appears in `POSITIVE_WORDS`, then sums them.
It is the same conditional logic from Module 3, expressed as a generator inside `sum()`.
